In [0]:
CATALOG = "supplychainiq"
SOURCE  = "/Volumes/supplychainiq/bronze/landing/pos/"
CHECKPOINT = "/Volumes/supplychainiq/ops/checkpoints/bronze_pos"
TARGET  = f"{CATALOG}.bronze.pos_events"

In [0]:
from pyspark.sql import functions as F

before = spark.table(TARGET).count() if spark.catalog.tableExists(TARGET) else 0

query = (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", f"{CHECKPOINT}/schema")
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .option("rescuedDataColumn", "_rescued_data")
    .load(SOURCE)
    .select("*",
            F.col("_metadata.file_path").alias("_source_file"),
            F.col("_metadata.file_modification_time").alias("_file_modified_at"),
            F.current_timestamp().alias("_ingested_at"))
    .writeStream
    .option("checkpointLocation", CHECKPOINT)
    .trigger(availableNow=True)
    .toTable(TARGET)
)
query.awaitTermination()

after = spark.table(TARGET).count()
print(f"{after - before:,} rows ingested this run (table total: {after:,})")

In [0]:
%sql
SELECT COUNT(*)                         AS rows,
       COUNT(DISTINCT event_id)         AS distinct_events,
       COUNT(*) - COUNT(DISTINCT event_id) AS duplicate_rows,
       COUNT(DISTINCT _source_file)     AS files,
       MIN(event_date), MAX(event_date)
FROM supplychainiq.bronze.pos_events;

In [0]:
%sql
SELECT DISTINCT
  regexp_extract(_source_file, 'pos_[A-Z]+_[0-9]+_([0-9-]+)[.]jsonl', 1) AS file_event_date,
  regexp_extract(_source_file, 'pos/([0-9-]+)/', 1)                    AS arrival_folder,
  store_id
FROM supplychainiq.bronze.pos_events
WHERE regexp_extract(_source_file, 'pos_[A-Z]+_[0-9]+_([0-9-]+)[.]jsonl', 1)
   <> regexp_extract(_source_file, 'pos/([0-9-]+)/', 1)
ORDER BY file_event_date;

In [0]:
%sql
SELECT DISTINCT
  regexp_extract(_source_file, 'pos_[A-Z]+_[0-9]+_([0-9-]+)[.]jsonl', 1) AS file_event_date,
  regexp_extract(_source_file, 'pos/([0-9-]+)/', 1)                    AS arrival_folder,
  _source_file
FROM supplychainiq.bronze.pos_events LIMIT 3;